In [20]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch import optim

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

In [21]:

manifest = "../data/realwaste_data.csv"
dataset_root = "../data/RealWaste"

print(manifest)
print(dataset_root)

../data/realwaste_data.csv
../data/RealWaste


In [22]:
class RealWasteDataset(Dataset):

    def __init__(self, data_file, image_path, transform=None):
        self.image_path = Path(image_path)
        self.image_labels = pd.read_csv(data_file)
        self.transform = transform

        pairs = (
            self.image_labels[["label", "class_name"]]
            .drop_duplicates()
            .sort_values("label")
        )

        self.classes = pairs.class_name.tolist()
        self.class_to_idx = dict(zip(pairs.class_name, pairs.label))

    def __len__(self):
        return len(self.image_labels)

    def __getitem__(self, index):
        row = self.image_labels.iloc[index]

        image = Image.open(
            self.image_path / row.filepath
        ).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, int(row.label)

    @property
    def targets(self):
        return self.image_labels.label.tolist()

In [23]:
base = RealWasteDataset(
    manifest,
    dataset_root
)

print("Total images:", len(base))
print("Classes:", base.classes)

Total images: 4752
Classes: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']


In [24]:
train_tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [25]:
y = base.targets

idx_train, idx_temp = train_test_split(
    range(len(y)),
    test_size=0.3,
    stratify=y,
    random_state=42
)

y_temp = [y[i] for i in idx_temp]

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", len(idx_train))
print("Validation:", len(idx_val))
print("Test:", len(idx_test))

Training: 3326
Validation: 713
Test: 713


In [26]:
train_ds = Subset(
    RealWasteDataset(manifest, dataset_root, transform=train_tf),
    idx_train
)

val_ds = Subset(
    RealWasteDataset(manifest, dataset_root, transform=eval_tf),
    idx_val
)

test_ds = Subset(
    RealWasteDataset(manifest, dataset_root, transform=eval_tf),
    idx_test
)

print(len(train_ds))
print(len(val_ds))
print(len(test_ds))

3326
713
713


In [27]:
train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False
)

print("DataLoaders created.")

DataLoaders created.


In [28]:
class WasteImageNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(4096, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 9)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)

model = WasteImageNN().to(device)

print(model)

Using: cpu
WasteImageNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=4096, out_features=256, bias=True)
    (3): ReLU(inplace

In [30]:
total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Total parameters: {total_params:,}")

Total parameters: 1,439,561


In [31]:
def train_model(model, train_loader, val_loader, epochs, lr, optimizer_name):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Loss function
    criterion = nn.CrossEntropyLoss()

    # Choose optimizer
    if optimizer_name == "Adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr
        )

    elif optimizer_name == "SGD":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr
        )

    elif optimizer_name == "SGD_Momentum":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9
        )

    else:
        raise ValueError("Unknown optimizer")

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # Training
        model.train()
        running_train_loss = 0.0

        for batch_x, batch_y in train_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            outputs = model(batch_x)

            loss = criterion(outputs, batch_y)

            loss.backward()

            optimizer.step()

            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # validation
        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():

            for batch_x, batch_y in val_loader:

                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)

                outputs = model(batch_x)

                loss = criterion(outputs, batch_y)

                running_val_loss += loss.item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )

    return train_losses, val_losses

In [34]:
model_test = WasteImageNN()

train_losses, val_losses = train_model(
    model=model_test,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.001,
    optimizer_name="Adam"
)

Epoch [1/20] | Train Loss: 2.1169 | Val Loss: 1.9529
Epoch [2/20] | Train Loss: 1.8004 | Val Loss: 1.5976
Epoch [3/20] | Train Loss: 1.5767 | Val Loss: 1.4587
Epoch [4/20] | Train Loss: 1.4384 | Val Loss: 1.3613
Epoch [5/20] | Train Loss: 1.3306 | Val Loss: 1.2287
Epoch [6/20] | Train Loss: 1.2197 | Val Loss: 1.1892
Epoch [7/20] | Train Loss: 1.1681 | Val Loss: 1.2284
Epoch [8/20] | Train Loss: 1.0969 | Val Loss: 1.0087
Epoch [9/20] | Train Loss: 1.0462 | Val Loss: 0.9987
Epoch [10/20] | Train Loss: 0.9767 | Val Loss: 1.0209
Epoch [11/20] | Train Loss: 0.9629 | Val Loss: 0.9658
Epoch [12/20] | Train Loss: 0.8962 | Val Loss: 0.8742
Epoch [13/20] | Train Loss: 0.8973 | Val Loss: 0.9125
Epoch [14/20] | Train Loss: 0.8386 | Val Loss: 0.9169
Epoch [15/20] | Train Loss: 0.8134 | Val Loss: 0.8271
Epoch [16/20] | Train Loss: 0.7851 | Val Loss: 0.8887
Epoch [17/20] | Train Loss: 0.7939 | Val Loss: 0.8312
Epoch [18/20] | Train Loss: 0.7257 | Val Loss: 0.7756
Epoch [19/20] | Train Loss: 0.6762 | 

In [36]:
def evaluate_test(model, test_loader):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for batch_x, batch_y in test_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)

            _, predicted = torch.max(outputs, 1)

            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    test_accuracy = 100 * correct / total

    print(f"TEST ACCURACY: {test_accuracy:.2f}%")

    return test_accuracy

In [37]:
test_acc = evaluate_test(
    model=model_test,
    test_loader=test_loader
)

TEST ACCURACY: 69.14%


In [38]:
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    model_test.state_dict(),
    models_dir / "model_adam_0.001.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [39]:
model_adam_01 = WasteImageNN()

train_losses_adam_01, val_losses_adam_01 = train_model(
    model=model_adam_01,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.01,
    optimizer_name="Adam"
)


Epoch [1/20] | Train Loss: 3.1654 | Val Loss: 2.8849
Epoch [2/20] | Train Loss: 2.1529 | Val Loss: 2.0339
Epoch [3/20] | Train Loss: 2.0816 | Val Loss: 1.9864
Epoch [4/20] | Train Loss: 2.0452 | Val Loss: 1.9756
Epoch [5/20] | Train Loss: 2.0199 | Val Loss: 1.9471
Epoch [6/20] | Train Loss: 1.9753 | Val Loss: 1.9024
Epoch [7/20] | Train Loss: 1.9598 | Val Loss: 1.8989
Epoch [8/20] | Train Loss: 1.9530 | Val Loss: 1.8743
Epoch [9/20] | Train Loss: 1.9383 | Val Loss: 1.8551
Epoch [10/20] | Train Loss: 1.9099 | Val Loss: 1.8386
Epoch [11/20] | Train Loss: 1.8966 | Val Loss: 1.9011
Epoch [12/20] | Train Loss: 1.8953 | Val Loss: 1.8474
Epoch [13/20] | Train Loss: 1.8745 | Val Loss: 1.8074
Epoch [14/20] | Train Loss: 1.8549 | Val Loss: 1.7943
Epoch [15/20] | Train Loss: 1.8508 | Val Loss: 1.7839
Epoch [16/20] | Train Loss: 1.8413 | Val Loss: 1.7711
Epoch [17/20] | Train Loss: 1.8401 | Val Loss: 1.8128
Epoch [18/20] | Train Loss: 1.8161 | Val Loss: 1.7322
Epoch [19/20] | Train Loss: 1.8295 | 

In [40]:
test_acc_adam_01 = evaluate_test(
    model=model_adam_01,
    test_loader=test_loader
)

TEST ACCURACY: 36.04%


In [41]:
torch.save(
    model_adam_01.state_dict(),
    models_dir / "model_adam_0.01.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [42]:
model_test = WasteImageNN()

train_losses, val_losses = train_model(
    model=model_test,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.0001,
    optimizer_name="Adam"
)

Epoch [1/20] | Train Loss: 2.1422 | Val Loss: 2.0612
Epoch [2/20] | Train Loss: 1.9882 | Val Loss: 1.8635
Epoch [3/20] | Train Loss: 1.8431 | Val Loss: 1.7260
Epoch [4/20] | Train Loss: 1.7403 | Val Loss: 1.6628
Epoch [5/20] | Train Loss: 1.6263 | Val Loss: 1.5610
Epoch [6/20] | Train Loss: 1.5647 | Val Loss: 1.5090
Epoch [7/20] | Train Loss: 1.5091 | Val Loss: 1.4454
Epoch [8/20] | Train Loss: 1.4558 | Val Loss: 1.4159
Epoch [9/20] | Train Loss: 1.4089 | Val Loss: 1.4072
Epoch [10/20] | Train Loss: 1.3709 | Val Loss: 1.3305
Epoch [11/20] | Train Loss: 1.3524 | Val Loss: 1.3034
Epoch [12/20] | Train Loss: 1.3281 | Val Loss: 1.3078
Epoch [13/20] | Train Loss: 1.2708 | Val Loss: 1.2494
Epoch [14/20] | Train Loss: 1.2587 | Val Loss: 1.2323
Epoch [15/20] | Train Loss: 1.2261 | Val Loss: 1.2250
Epoch [16/20] | Train Loss: 1.1975 | Val Loss: 1.1834
Epoch [17/20] | Train Loss: 1.2001 | Val Loss: 1.1834
Epoch [18/20] | Train Loss: 1.1703 | Val Loss: 1.1437
Epoch [19/20] | Train Loss: 1.1519 | 

In [46]:
test_acc_adam_0001 = evaluate_test(
    model=model_test,
    test_loader=test_loader
)

TEST ACCURACY: 57.92%


In [47]:
torch.save(
    model_test.state_dict(),
    models_dir / "model_adam_0.0001.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [48]:
model_sgd_01 = WasteImageNN()

train_losses_sgd_01, val_losses_sgd_01 = train_model(
    model=model_sgd_01,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.01,
    optimizer_name="SGD"
)

Epoch [1/20] | Train Loss: 2.1823 | Val Loss: 2.1608
Epoch [2/20] | Train Loss: 2.1596 | Val Loss: 2.1334
Epoch [3/20] | Train Loss: 2.1443 | Val Loss: 2.1217
Epoch [4/20] | Train Loss: 2.1411 | Val Loss: 2.1148
Epoch [5/20] | Train Loss: 2.1356 | Val Loss: 2.1045
Epoch [6/20] | Train Loss: 2.1263 | Val Loss: 2.0934
Epoch [7/20] | Train Loss: 2.1082 | Val Loss: 2.0725
Epoch [8/20] | Train Loss: 2.0861 | Val Loss: 2.0448
Epoch [9/20] | Train Loss: 2.0555 | Val Loss: 1.9930
Epoch [10/20] | Train Loss: 2.0024 | Val Loss: 1.9292
Epoch [11/20] | Train Loss: 1.9432 | Val Loss: 1.8870
Epoch [12/20] | Train Loss: 1.9092 | Val Loss: 1.8514
Epoch [13/20] | Train Loss: 1.8642 | Val Loss: 1.7939
Epoch [14/20] | Train Loss: 1.8324 | Val Loss: 1.7626
Epoch [15/20] | Train Loss: 1.7984 | Val Loss: 1.7288
Epoch [16/20] | Train Loss: 1.7711 | Val Loss: 1.7048
Epoch [17/20] | Train Loss: 1.7510 | Val Loss: 1.6827
Epoch [18/20] | Train Loss: 1.7300 | Val Loss: 1.6877
Epoch [19/20] | Train Loss: 1.7061 | 

In [49]:
test_acc_sgd_01 = evaluate_test(
    model=model_sgd_01,
    test_loader=test_loader
)

TEST ACCURACY: 43.90%


In [50]:
torch.save(
    model_sgd_01.state_dict(),
    models_dir / "model_sgd_0.01.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [51]:
model_sgd_001 = WasteImageNN()

train_losses_sgd_001, val_losses_sgd_001 = train_model(
    model=model_sgd_001,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.001,
    optimizer_name="SGD"
)

Epoch [1/20] | Train Loss: 2.2004 | Val Loss: 2.2004
Epoch [2/20] | Train Loss: 2.1992 | Val Loss: 2.1984
Epoch [3/20] | Train Loss: 2.1967 | Val Loss: 2.1964
Epoch [4/20] | Train Loss: 2.1956 | Val Loss: 2.1943
Epoch [5/20] | Train Loss: 2.1944 | Val Loss: 2.1923
Epoch [6/20] | Train Loss: 2.1925 | Val Loss: 2.1904
Epoch [7/20] | Train Loss: 2.1916 | Val Loss: 2.1884
Epoch [8/20] | Train Loss: 2.1891 | Val Loss: 2.1864
Epoch [9/20] | Train Loss: 2.1881 | Val Loss: 2.1844
Epoch [10/20] | Train Loss: 2.1861 | Val Loss: 2.1824
Epoch [11/20] | Train Loss: 2.1847 | Val Loss: 2.1804
Epoch [12/20] | Train Loss: 2.1829 | Val Loss: 2.1784
Epoch [13/20] | Train Loss: 2.1822 | Val Loss: 2.1764
Epoch [14/20] | Train Loss: 2.1798 | Val Loss: 2.1743
Epoch [15/20] | Train Loss: 2.1783 | Val Loss: 2.1722
Epoch [16/20] | Train Loss: 2.1763 | Val Loss: 2.1700
Epoch [17/20] | Train Loss: 2.1755 | Val Loss: 2.1679
Epoch [18/20] | Train Loss: 2.1737 | Val Loss: 2.1657
Epoch [19/20] | Train Loss: 2.1718 | 

In [52]:
test_acc_sgd_001 = evaluate_test(
    model=model_sgd_001,
    test_loader=test_loader
)

TEST ACCURACY: 19.35%


In [53]:
torch.save(
    model_sgd_001.state_dict(),
    models_dir / "model_sgd_0.001.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [54]:
model_sgd_0001 = WasteImageNN()

train_losses_sgd_0001, val_losses_sgd_0001 = train_model(
    model=model_sgd_0001,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.0001,
    optimizer_name="SGD"
)

Epoch [1/20] | Train Loss: 2.1926 | Val Loss: 2.1921
Epoch [2/20] | Train Loss: 2.1925 | Val Loss: 2.1919
Epoch [3/20] | Train Loss: 2.1922 | Val Loss: 2.1917
Epoch [4/20] | Train Loss: 2.1920 | Val Loss: 2.1915
Epoch [5/20] | Train Loss: 2.1925 | Val Loss: 2.1912
Epoch [6/20] | Train Loss: 2.1920 | Val Loss: 2.1910
Epoch [7/20] | Train Loss: 2.1919 | Val Loss: 2.1908
Epoch [8/20] | Train Loss: 2.1918 | Val Loss: 2.1906
Epoch [9/20] | Train Loss: 2.1917 | Val Loss: 2.1904
Epoch [10/20] | Train Loss: 2.1912 | Val Loss: 2.1902
Epoch [11/20] | Train Loss: 2.1915 | Val Loss: 2.1900
Epoch [12/20] | Train Loss: 2.1911 | Val Loss: 2.1898
Epoch [13/20] | Train Loss: 2.1910 | Val Loss: 2.1896
Epoch [14/20] | Train Loss: 2.1907 | Val Loss: 2.1894
Epoch [15/20] | Train Loss: 2.1911 | Val Loss: 2.1891
Epoch [16/20] | Train Loss: 2.1901 | Val Loss: 2.1889
Epoch [17/20] | Train Loss: 2.1903 | Val Loss: 2.1887
Epoch [18/20] | Train Loss: 2.1898 | Val Loss: 2.1885
Epoch [19/20] | Train Loss: 2.1896 | 

In [55]:
test_acc_sgd_0001 = evaluate_test(
    model=model_sgd_0001,
    test_loader=test_loader
)

TEST ACCURACY: 12.76%


In [56]:
torch.save(
    model_sgd_0001.state_dict(),
    models_dir / "model_sgd_0.0001.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [57]:
model_momentum_01 = WasteImageNN()

train_losses_momentum_01, val_losses_momentum_01 = train_model(
    model=model_momentum_01,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.01,
    optimizer_name="SGD_Momentum"
)

Epoch [1/20] | Train Loss: 2.1638 | Val Loss: 2.1249
Epoch [2/20] | Train Loss: 2.0992 | Val Loss: 2.0359
Epoch [3/20] | Train Loss: 1.9459 | Val Loss: 1.8444
Epoch [4/20] | Train Loss: 1.8462 | Val Loss: 1.7676
Epoch [5/20] | Train Loss: 1.6879 | Val Loss: 1.6667
Epoch [6/20] | Train Loss: 1.6063 | Val Loss: 1.5362
Epoch [7/20] | Train Loss: 1.5507 | Val Loss: 1.4448
Epoch [8/20] | Train Loss: 1.5002 | Val Loss: 1.4544
Epoch [9/20] | Train Loss: 1.4317 | Val Loss: 1.3762
Epoch [10/20] | Train Loss: 1.4053 | Val Loss: 1.3782
Epoch [11/20] | Train Loss: 1.3459 | Val Loss: 1.3239
Epoch [12/20] | Train Loss: 1.3099 | Val Loss: 1.3191
Epoch [13/20] | Train Loss: 1.3145 | Val Loss: 1.2947
Epoch [14/20] | Train Loss: 1.2643 | Val Loss: 1.2355
Epoch [15/20] | Train Loss: 1.2190 | Val Loss: 1.2195
Epoch [16/20] | Train Loss: 1.1723 | Val Loss: 1.1444
Epoch [17/20] | Train Loss: 1.1799 | Val Loss: 1.1368
Epoch [18/20] | Train Loss: 1.1268 | Val Loss: 1.0910
Epoch [19/20] | Train Loss: 1.1403 | 

In [58]:
test_acc_momentum_01 = evaluate_test(
    model=model_momentum_01,
    test_loader=test_loader
)

TEST ACCURACY: 60.17%


In [59]:
torch.save(
    model_momentum_01.state_dict(),
    models_dir / "model_sgd_momentum_0.01.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [60]:
model_momentum_001 = WasteImageNN()

train_losses_momentum_001, val_losses_momentum_001 = train_model(
    model=model_momentum_001,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.001,
    optimizer_name="SGD_Momentum"
)

Epoch [1/20] | Train Loss: 2.1915 | Val Loss: 2.1794
Epoch [2/20] | Train Loss: 2.1742 | Val Loss: 2.1572
Epoch [3/20] | Train Loss: 2.1576 | Val Loss: 2.1353
Epoch [4/20] | Train Loss: 2.1469 | Val Loss: 2.1232
Epoch [5/20] | Train Loss: 2.1409 | Val Loss: 2.1182
Epoch [6/20] | Train Loss: 2.1332 | Val Loss: 2.1122
Epoch [7/20] | Train Loss: 2.1320 | Val Loss: 2.1054
Epoch [8/20] | Train Loss: 2.1278 | Val Loss: 2.0984
Epoch [9/20] | Train Loss: 2.1162 | Val Loss: 2.0863
Epoch [10/20] | Train Loss: 2.1015 | Val Loss: 2.0644
Epoch [11/20] | Train Loss: 2.0797 | Val Loss: 2.0349
Epoch [12/20] | Train Loss: 2.0428 | Val Loss: 1.9816
Epoch [13/20] | Train Loss: 1.9869 | Val Loss: 1.9093
Epoch [14/20] | Train Loss: 1.9209 | Val Loss: 1.8359
Epoch [15/20] | Train Loss: 1.8710 | Val Loss: 1.7935
Epoch [16/20] | Train Loss: 1.8408 | Val Loss: 1.7662
Epoch [17/20] | Train Loss: 1.8044 | Val Loss: 1.7345
Epoch [18/20] | Train Loss: 1.7620 | Val Loss: 1.7013
Epoch [19/20] | Train Loss: 1.7405 | 

In [61]:
test_acc_momentum_001 = evaluate_test(
    model=model_momentum_001,
    test_loader=test_loader
)

TEST ACCURACY: 42.50%


In [62]:
torch.save(
    model_momentum_001.state_dict(),
    models_dir / "model_sgd_momentum_0.001.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [63]:
model_momentum_0001 = WasteImageNN()

train_losses_momentum_0001, val_losses_momentum_0001 = train_model(
    model=model_momentum_0001,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=20,
    lr=0.0001,
    optimizer_name="SGD_Momentum"
)

Epoch [1/20] | Train Loss: 2.2001 | Val Loss: 2.1980
Epoch [2/20] | Train Loss: 2.1976 | Val Loss: 2.1960
Epoch [3/20] | Train Loss: 2.1967 | Val Loss: 2.1940
Epoch [4/20] | Train Loss: 2.1948 | Val Loss: 2.1921
Epoch [5/20] | Train Loss: 2.1935 | Val Loss: 2.1903
Epoch [6/20] | Train Loss: 2.1926 | Val Loss: 2.1885
Epoch [7/20] | Train Loss: 2.1906 | Val Loss: 2.1867
Epoch [8/20] | Train Loss: 2.1894 | Val Loss: 2.1849
Epoch [9/20] | Train Loss: 2.1869 | Val Loss: 2.1831
Epoch [10/20] | Train Loss: 2.1848 | Val Loss: 2.1813
Epoch [11/20] | Train Loss: 2.1842 | Val Loss: 2.1793
Epoch [12/20] | Train Loss: 2.1826 | Val Loss: 2.1775
Epoch [13/20] | Train Loss: 2.1817 | Val Loss: 2.1755
Epoch [14/20] | Train Loss: 2.1796 | Val Loss: 2.1736
Epoch [15/20] | Train Loss: 2.1785 | Val Loss: 2.1716
Epoch [16/20] | Train Loss: 2.1766 | Val Loss: 2.1696
Epoch [17/20] | Train Loss: 2.1752 | Val Loss: 2.1676
Epoch [18/20] | Train Loss: 2.1721 | Val Loss: 2.1654
Epoch [19/20] | Train Loss: 2.1720 | 

In [64]:
test_acc_momentum_0001 = evaluate_test(
    model=model_momentum_0001,
    test_loader=test_loader
)

TEST ACCURACY: 19.35%


In [65]:
torch.save(
    model_momentum_001.state_dict(),
    models_dir / "model_sgd_momentum_0.0001.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [66]:
print(model_test)

WasteImageNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=4096, out_features=256, bias=True)
    (3): ReLU(inplace=True)
    

In [68]:
# Create a fresh Model A
model_adam_001 = WasteImageNN()

# Load the saved Adam 0.001 weights
model_adam_001.load_state_dict(
    torch.load(
        "../models/model_adam_0.001.pth",
        map_location="cpu"
    )
)

print("Adam 0.001 Model A loaded successfully!")

Adam 0.001 Model A loaded successfully!


In [70]:
class_names = [
    "Cardboard",
    "Food Organics",
    "Glass",
    "Metal",
    "Miscellaneous Trash",
    "Paper",
    "Plastic",
    "Textile Trash",
    "Vegetation"
]

print(class_names)

['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']


In [71]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_adam_001 = model_adam_001.to(device)
model_adam_001.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        outputs = model_adam_001(batch_x)

        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())


# Confusion Matrix
cm = confusion_matrix(all_labels, all_predictions)

print("Confusion Matrix:")
print(cm)


# Overall Precision
precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted"
)

# Overall Recall
recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print(f"\nWeighted Precision: {precision:.4f}")
print(f"Weighted Recall: {recall:.4f}")


# Classification Report
print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)

Confusion Matrix:
[[54  2  2  5  2  2  1  0  1]
 [ 2 39  0  3  2  0  3  0 13]
 [ 2  0 51  4  3  0  3  0  0]
 [ 2  1  2 90  9  5  7  2  0]
 [ 3  3  3 12 30  5  8  6  5]
 [ 5  0  4  7  1 54  3  1  0]
 [ 3  1 14 20 11  1 84  4  0]
 [ 0  0  1  6  4  8  0 28  1]
 [ 0  0  0  0  1  0  0  1 63]]

Weighted Precision: 0.6962
Weighted Recall: 0.6914

Classification Report:
                     precision    recall  f1-score   support

          Cardboard     0.7606    0.7826    0.7714        69
      Food Organics     0.8478    0.6290    0.7222        62
              Glass     0.6623    0.8095    0.7286        63
              Metal     0.6122    0.7627    0.6792       118
Miscellaneous Trash     0.4762    0.4000    0.4348        75
              Paper     0.7200    0.7200    0.7200        75
            Plastic     0.7706    0.6087    0.6802       138
      Textile Trash     0.6667    0.5833    0.6222        48
         Vegetation     0.7590    0.9692    0.8514        65

           accuracy    